# Capítulo 7 — Como Saber se os Grupos São Bons? Avaliação e Validação

Códigos do capítulo. Execute as células em ordem; cada bloco é independente dos demais.


## 7.2 Antes de tudo: existe estrutura nos dados?

In [ ]:
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.datasets import make_blobs

def hopkins(X, m=None, seed=42):
    rng = np.random.default_rng(seed)
    n, d = X.shape
    m = m or max(10, int(0.1 * n))
    nn = NearestNeighbors(n_neighbors=2).fit(X)
    idx = rng.choice(n, m, replace=False)
    w = nn.kneighbors(X[idx], 2)[0][:, 1]          # vizinho mais próximo (exclui ele mesmo)
    U = rng.uniform(X.min(0), X.max(0), (m, d))
    u = nn.kneighbors(U, 1)[0][:, 0]
    return (u ** d).sum() / ((u ** d).sum() + (w ** d).sum())

rng = np.random.default_rng(0)
uniforme = rng.random((500, 2))
agrupado, _ = make_blobs(500, centers=4, random_state=2)
nuvem = rng.normal(size=(500, 2))                  # uma única nuvem gaussiana: sem grupos

print("Hopkins uniforme:", round(hopkins(uniforme), 2))
print("Hopkins agrupado:", round(hopkins(agrupado), 2))
print("Hopkins uma nuvem:", round(hopkins(nuvem), 2))

## 7.3.2 O coeficiente de silhueta

In [ ]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples

X, _ = make_blobs(n_samples=500, centers=4, random_state=8)

for k in range(2, 8):
    rot = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)
    print(f"k={k}  silhueta média = {silhouette_score(X, rot):.3f}")

# Silhueta individual de cada ponto
rot = KMeans(n_clusters=4, n_init=10, random_state=0).fit_predict(X)
s = silhouette_samples(X, rot)
print("Pontos com silhueta negativa:", (s < 0).sum())

## 7.3.5 Índice de Dunn

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score

X, _ = make_blobs(n_samples=500, centers=4, random_state=8)
print(" k  silhueta   CH       DB")
for k in range(2, 8):
    r = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)
    sil = silhouette_score(X, r)
    ch = calinski_harabasz_score(X, r)
    db = davies_bouldin_score(X, r)
    print(f"{k:2d}  {sil:7.3f}  {ch:8.1f}  {db:6.3f}")

## 7.4.2 Estabilidade

In [ ]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

X, _ = make_blobs(n_samples=600, centers=4, random_state=8)
rng = np.random.default_rng(0)

def estabilidade(k, repeticoes=20):
    scores = []
    for _ in range(repeticoes):
        i1 = rng.choice(len(X), len(X) // 2, replace=False)
        i2 = rng.choice(len(X), len(X) // 2, replace=False)
        m1 = KMeans(k, n_init=5, random_state=0).fit(X[i1])
        m2 = KMeans(k, n_init=5, random_state=1).fit(X[i2])
        # Compara as duas soluções no conjunto inteiro
        scores.append(adjusted_rand_score(m1.predict(X), m2.predict(X)))
    return np.mean(scores)

for k in range(2, 8):
    print(f"k={k}  estabilidade (ARI médio) = {estabilidade(k):.3f}")

## 7.5.5 Homogeneidade, completude e V-measure

In [ ]:
from sklearn.datasets import load_iris
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import (adjusted_rand_score, normalized_mutual_info_score,
                             homogeneity_completeness_v_measure)

X, y = load_iris(return_X_y=True)
for nome, modelo in [("K-means", KMeans(3, n_init=10, random_state=0)),
                     ("Ward   ", AgglomerativeClustering(3, linkage="ward"))]:
    r = modelo.fit_predict(X)
    h, c, v = homogeneity_completeness_v_measure(y, r)
    print(f"{nome} ARI={adjusted_rand_score(y, r):.3f}  "
          f"NMI={normalized_mutual_info_score(y, r):.3f}  "
          f"H={h:.3f} C={c:.3f} V={v:.3f}")

## 7.6.1 Perfil dos grupos

In [ ]:
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

dados = load_wine(as_frame=True)
X = dados.data[["alcohol", "malic_acid", "flavanoids", "color_intensity"]]
grupos = KMeans(3, n_init=10, random_state=0).fit_predict(StandardScaler().fit_transform(X))

perfil = X.assign(grupo=grupos).groupby("grupo").mean()
relativo = (perfil / X.mean()).round(2)      # >1: acima da média geral
print(relativo)

## Resumo do capítulo